## Setup Environment (Kaggle / Colab / W&B)

In [ ]:
import io
import os
import sys
import json
import random
import zipfile
import urllib.request
import importlib.util
import subprocess
import yaml
import numpy as np
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from seqeval.metrics import classification_report
from seqeval.scheme import IOB2

REPO_URL = "https://github.com/TTins1702/NER-PhoNER_COVID19-UET.git"
REPO_DIR = "NER-PhoNER_COVID19-UET"

if os.path.exists("../configs/phobert.yaml"):
    os.chdir("..")
if not os.path.exists("configs/phobert.yaml"):
    if os.path.exists(REPO_DIR):
        os.chdir(REPO_DIR)
        subprocess.run(["git", "pull"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL], check=True)
        os.chdir(REPO_DIR)
    if not os.path.exists("configs/phobert.yaml"):
        subprocess.run(["git", "fetch", "origin", "dangle"], check=False)
        subprocess.run(["git", "checkout", "dangle"], check=False)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

if "COLAB_RELEASE_TAG" in os.environ or "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    missing = [pkg for pkg in ["transformers", "seqeval", "pyyaml", "wandb"] if importlib.util.find_spec(pkg if pkg != "pyyaml" else "yaml") is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    if os.path.exists("requirements.txt"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)

if not os.path.exists("data/word/train_word.json"):
    if os.path.exists("src/data/download.py"):
        subprocess.run([sys.executable, "-m", "src.data.download", "--output-dir", "data"], check=True)
    else:
        zip_url = "https://github.com/VinAIResearch/PhoNER_COVID19/archive/refs/heads/main.zip"
        with urllib.request.urlopen(zip_url) as resp:
            raw_zip = resp.read()
        with zipfile.ZipFile(io.BytesIO(raw_zip)) as zf:
            for member in zf.infolist():
                if "/data/" in member.filename and not member.is_dir():
                    rel = member.filename.split("/data/", 1)[1]
                    target = os.path.join("data", rel)
                    os.makedirs(os.path.dirname(target), exist_ok=True)
                    with open(target, "wb") as out_f:
                        out_f.write(zf.read(member.filename))

wandb_key = os.environ.get("WANDB_API_KEY")
if not wandb_key:
    try:
        from kaggle_secrets import UserSecretsClient
        wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception:
        pass
if not wandb_key:
    try:
        from google.colab import userdata
        wandb_key = userdata.get("WANDB_API_KEY")
    except Exception:
        pass

try:
    import wandb
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
        os.environ.pop("WANDB_MODE", None)
        wandb.login(key=wandb_key)
    else:
        os.environ["WANDB_MODE"] = "offline"
except ImportError:
    wandb = None

with open("configs/phobert.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

from src.data.phobert_dataset import (
    ENTITY_TYPES,
    PhoBERTDataset,
    decode_word_level_predictions,
    extract_word_level_labels,
    get_label_maps,
    inspect_bpe_sample,
    phobert_collate_fn,
)
from src.models.phobert_ner import PhoBERTNER

print("Setup done. Device:", "cuda" if torch.cuda.is_available() else "cpu")

## Week 1: BPE Tokenizer Check

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"])
sample_words = ["Bệnh_nhân", "ở", "Vĩnh_Phúc", "bị", "sốt", "cao", "."]
for word, subwords in inspect_bpe_sample(tokenizer, sample_words):
    print(f"{word} -> {subwords}")

## Week 2: Dataset, Alignment Check & 1-Epoch Baseline

In [ ]:
label2id, id2label = get_label_maps()
level = cfg["data_level"]

train_ds = PhoBERTDataset(
    os.path.join(cfg["data_dir"], level, f"train_{level}.json"),
    tokenizer,
    label2id=label2id,
    max_seq_length=cfg["max_seq_length"],
)
dev_ds = PhoBERTDataset(
    os.path.join(cfg["data_dir"], level, f"dev_{level}.json"),
    tokenizer,
    label2id=label2id,
    max_seq_length=cfg["max_seq_length"],
)
test_ds = PhoBERTDataset(
    os.path.join(cfg["data_dir"], level, f"test_{level}.json"),
    tokenizer,
    label2id=label2id,
    max_seq_length=cfg["max_seq_length"],
)

for name, ds in [("train", train_ds), ("dev", dev_ds), ("test", test_ds)]:
    print(name, ds.get_truncation_stats())

first_item = train_ds[0]
print("Alignment match on sample 0:", int(first_item["word_mask"].sum().item()) == len(train_ds.samples[0]["words"]))

collate_fn = lambda batch: phobert_collate_fn(batch, pad_token_id=tokenizer.pad_token_id)
sample_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=False, collate_fn=collate_fn)
sample_batch = next(iter(sample_loader))
print("Batch tensor shapes:", {k: tuple(v.shape) for k, v in sample_batch.items() if isinstance(v, torch.Tensor)})

RUN_WEEK2_SANITY = False

## Week 3: Training & Strict Word-Level Evaluation Functions

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def to_native(obj):
    if isinstance(obj, dict):
        return {k: to_native(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_native(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_native(v) for v in obj]
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    return obj


def count_bio_violations(tag_sequences):
    violations = 0
    for seq in tag_sequences:
        prev_type = None
        for tag in seq:
            if tag.startswith("I-"):
                cur_type = tag[2:]
                if prev_type != cur_type:
                    violations += 1
                prev_type = cur_type
            elif tag.startswith("B-"):
                prev_type = tag[2:]
            else:
                prev_type = None
    return violations


def build_standardized_metrics(report, y_pred):
    per_entity = {}
    for entity_type in ENTITY_TYPES:
        entry = report.get(entity_type, {})
        per_entity[entity_type] = {
            "precision": float(entry.get("precision", 0.0)),
            "recall": float(entry.get("recall", 0.0)),
            "f1": float(entry.get("f1-score", 0.0)),
            "support": int(entry.get("support", 0)),
        }
    micro = report.get("micro avg", {})
    macro = report.get("macro avg", {})
    return {
        "micro_precision": float(micro.get("precision", 0.0)),
        "micro_recall": float(micro.get("recall", 0.0)),
        "micro_f1": float(micro.get("f1-score", 0.0)),
        "macro_f1": float(macro.get("f1-score", 0.0)),
        "per_entity": per_entity,
        "bio_violations": count_bio_violations(y_pred),
    }


def evaluate_model(model, loader, id2label, device):
    model.eval()
    y_true = []
    y_pred = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            word_mask = batch["word_mask"].to(device)
            pred_ids = model.decode(input_ids, attention_mask, word_mask)
            y_true.extend(extract_word_level_labels(batch["labels"], batch["word_mask"], id2label))
            y_pred.extend(decode_word_level_predictions(pred_ids, batch["word_mask"], id2label))
    report = to_native(
        classification_report(
            y_true,
            y_pred,
            mode="strict",
            scheme=IOB2,
            output_dict=True,
            zero_division=0,
        )
    )
    return report, build_standardized_metrics(report, y_pred), y_true, y_pred


def run_experiment(base_cfg, overrides=None):
    exp_cfg = {k: v for k, v in base_cfg.items() if k != "ablations"}
    if overrides:
        exp_cfg.update(overrides)

    set_seed(exp_cfg["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader = DataLoader(train_ds, batch_size=exp_cfg["batch_size"], shuffle=True, collate_fn=collate_fn)
    dev_loader = DataLoader(dev_ds, batch_size=exp_cfg["batch_size"], shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_ds, batch_size=exp_cfg["batch_size"], shuffle=False, collate_fn=collate_fn)

    model = PhoBERTNER(
        num_tags=len(label2id),
        model_name=exp_cfg["model_name"],
        feature_mode=exp_cfg["feature_mode"],
        use_crf=exp_cfg["use_crf"],
        freeze_backbone=exp_cfg["freeze_backbone"],
        dropout=exp_cfg["dropout"],
    ).to(device)

    optimizer = AdamW(
        model.build_optimizer_params(
            backbone_lr=float(exp_cfg["backbone_lr"]),
            head_lr=float(exp_cfg["head_lr"]),
            weight_decay=float(exp_cfg["weight_decay"]),
        )
    )
    total_steps = len(train_loader) * exp_cfg["epochs"]
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * exp_cfg["warmup_ratio"]),
        num_training_steps=max(1, total_steps),
    )

    os.makedirs(exp_cfg["checkpoint_dir"], exist_ok=True)
    os.makedirs(exp_cfg["output_dir"], exist_ok=True)
    exp_name = exp_cfg["experiment_name"]
    ckpt_path = os.path.join(exp_cfg["checkpoint_dir"], f"{exp_name}.pt")
    out_path = os.path.join(exp_cfg["output_dir"], f"{exp_name}.json")

    wb_run = None
    if wandb is not None:
        wb_run = wandb.init(
            project=exp_cfg["wandb_project"],
            name=exp_name,
            config=exp_cfg,
            reinit=True,
        )

    best_f1 = -1.0
    best_epoch = 0
    best_dev_report = None
    best_dev_metrics = None
    history = []

    for epoch in range(1, exp_cfg["epochs"] + 1):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            out = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                word_mask=batch["word_mask"].to(device),
                labels=batch["labels"].to(device),
                decode=False,
            )
            loss = out["loss"]
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), exp_cfg["max_grad_norm"])
            optimizer.step()
            scheduler.step()
            total_loss += float(loss.item())

        train_loss = total_loss / max(1, len(train_loader))
        dev_report, dev_metrics, _, _ = evaluate_model(model, dev_loader, id2label, device)
        dev_f1 = dev_metrics["micro_f1"]
        print(f"[{exp_name}] Epoch {epoch}/{exp_cfg['epochs']} - loss: {train_loss:.4f} - dev_micro_f1: {dev_f1:.4f}")

        step_record = {
            "epoch": float(epoch),
            "train_loss": train_loss,
            "dev_micro_f1": dev_f1,
            "dev_macro_f1": dev_metrics["macro_f1"],
            "dev_bio_violations": float(dev_metrics["bio_violations"]),
        }
        history.append(step_record)
        if wb_run is not None:
            wb_run.log(step_record)

        if dev_f1 >= best_f1:
            best_f1 = dev_f1
            best_epoch = epoch
            best_dev_report = dev_report
            best_dev_metrics = dev_metrics
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "config": exp_cfg,
                    "vocab_state": None,
                    "best_epoch": best_epoch,
                    "best_dev_f1": best_f1,
                },
                ckpt_path,
            )

    ckpt_obj = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt_obj["model_state_dict"])
    test_report, test_metrics, y_true, y_pred = evaluate_model(model, test_loader, id2label, device)
    print(f"[{exp_name}] Test micro_f1: {test_metrics['micro_f1']:.4f} | macro_f1: {test_metrics['macro_f1']:.4f}")

    if wb_run is not None:
        wb_run.log({
            "final_dev_micro_f1": best_f1,
            "final_test_micro_f1": test_metrics["micro_f1"],
            "final_test_macro_f1": test_metrics["macro_f1"],
            "final_test_bio_violations": float(test_metrics["bio_violations"]),
        })
        wb_run.finish()

    predictions = []
    for sample, gold_tags, pred_tags in zip(test_ds.samples, y_true, y_pred):
        predictions.append({
            "words": sample["words"][:len(gold_tags)],
            "gold": gold_tags,
            "pred": pred_tags,
        })

    payload = {
        "experiment_name": exp_name,
        "best_epoch": best_epoch,
        "best_dev_f1": best_f1,
        "best_dev_micro_f1": best_f1,
        "checkpoint_path": ckpt_path,
        "history": history,
        "dev": best_dev_metrics,
        "test": test_metrics,
        "dev_report": best_dev_report,
        "test_report": test_report,
        "test_predictions": predictions,
        "config": exp_cfg,
    }
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    return payload

if RUN_WEEK2_SANITY:
    run_experiment(cfg, {"epochs": 1, "experiment_name": "phobert_week2_sanity"})

## Week 3: Main PhoBERT Fine-tuning Run

In [ ]:
main_result = run_experiment(cfg)
for label, scores in main_result["test_report"].items():
    if isinstance(scores, dict):
        print(f"{label:22s} P={scores['precision']:.4f} R={scores['recall']:.4f} F1={scores['f1-score']:.4f} (n={scores['support']})")

## Week 4: PhoBERT Ablation Study

In [ ]:
ablation_rows = []
for exp in cfg["ablations"]:
    try:
        res = run_experiment(cfg, exp)
        ablation_rows.append({
            "experiment": res["experiment_name"],
            "micro_f1": res["test"]["micro_f1"],
            "macro_f1": res["test"]["macro_f1"],
        })
    except ImportError as exc:
        print(f"Skipped {exp['experiment_name']}: {exc}")

for row in ablation_rows:
    print(f"{row['experiment']:30s} | micro_f1={row['micro_f1']:.4f} | macro_f1={row['macro_f1']:.4f}")

## Week 5: Qualitative Analysis (Long Entities & OOV/Rare Types)

In [ ]:
main_json_path = os.path.join(cfg["output_dir"], "phobert_full_last_softmax.json")
if os.path.exists(main_json_path):
    with open(main_json_path, "r", encoding="utf-8") as f:
        phobert_res = json.load(f)
    for entity_name in ["SYMPTOM_AND_DISEASE", "LOCATION"]:
        if entity_name in phobert_res["test"]["per_entity"]:
            print(entity_name, "F1:", round(phobert_res["test"]["per_entity"][entity_name]["f1"], 4))

    bilstm_path = os.path.join(cfg["output_dir"], "bilstm_crf_word.json")
    if os.path.exists(bilstm_path):
        with open(bilstm_path, "r", encoding="utf-8") as f:
            bilstm_res = json.load(f)
        print("BiLSTM-CRF test micro_f1:", round(bilstm_res["test"]["micro_f1"], 4))
        print("PhoBERT test micro_f1:", round(phobert_res["test"]["micro_f1"], 4))

    shown = 0
    for item in phobert_res.get("test_predictions", []):
        gold = item["gold"]
        pred = item["pred"]
        words = item["words"]
        span_len = 0
        has_long_entity = False
        for tag in gold:
            if tag.startswith("B-"):
                span_len = 1
            elif tag.startswith("I-") and span_len > 0:
                span_len += 1
            else:
                span_len = 0
            if span_len >= 3:
                has_long_entity = True
                break
        if has_long_entity:
            pairs = [(w, g, p) for w, g, p in zip(words, gold, pred) if g != "O" or p != "O"]
            print("Long-entity example:", pairs)
            shown += 1
            if shown >= 5:
                break
else:
    print("Run Week 3 first to generate", main_json_path)

## Week 6: Reproducibility & Output Summary

In [ ]:
if os.path.exists(cfg["output_dir"]):
    for fname in sorted(os.listdir(cfg["output_dir"])):
        if fname.endswith(".json"):
            with open(os.path.join(cfg["output_dir"], fname), "r", encoding="utf-8") as f:
                data = json.load(f)
            test_sec = data.get("test", {})
            micro = test_sec.get("micro_f1", data.get("test_report", {}).get("micro avg", {}).get("f1-score"))
            macro = test_sec.get("macro_f1", data.get("test_report", {}).get("macro avg", {}).get("f1-score"))
            print(f"{fname:35s} | micro_f1={micro} | macro_f1={macro}")